<a href="https://colab.research.google.com/github/MehakNaseem/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mostafa-elwardany/machine-learning-prac/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This lane is framed as a ranking task built on a binary classification signal.

The end deliverable FlyRank's content team needs isn't a label on every page —
it's a short, ordered queue of pages to review first, given limited review
capacity each sprint. So under the hood, the model produces a probability of
"declining" for each page (that part is classification), but the thing we
actually use and defend is the resulting sort order — who's #1 to review, who's
#50 — not raw accuracy across all pages.

This matters because accuracy would treat a page ranked #29 vs #31 as basically
the same, when in practice the content team can only act on the top handful.
Ranking (via a classification probability) is the framing that matches how the
output actually gets used.

In [18]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Rows:", df.shape[0], "| declining rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000 | declining rate: 0.542


The target is is_declining_label, defined as trend_direction == "down".

This is a defined-rule proxy, not a directly observed business outcome (like
"revenue from this page dropped next quarter" or "the client lost this page
from their top rankings"). trend_direction is itself computed from a trailing
window of past traffic data, so the label describes recent past behavior —
it does not guarantee the page will keep declining going forward.

Using a proxy like this is reasonable as a starting point because it's cheap,
observable, and consistently defined across every page. But it means any
claim the model makes should be phrased carefully — "flagged as matching a
past decline pattern," not "predicted to decline in the future" — since the
label was never validated against what actually happened next.

In [19]:
df[["trend_direction", "trend_pct", "is_declining_label"]].sample(5, random_state=1)

,trend_direction,trend_pct,is_declining_label
10747,down,-41.4,1
12573,down,-90.7,1
29676,down,-100.0,1
8856,down,-44.7,1
21098,up,89.2,0


Success metric: Precision@50.

Of the top 50 pages the model (or rule) ranks highest, what fraction are
actually declining by our label? This is the metric I can defend because it
matches the real constraint on the other end — the content team can only
review a fixed, small number of pages per sprint, not all 30,000. So what
matters isn't accuracy across every page (most of which nobody will ever
look at), it's "of the pages we actually act on, how many were worth it."

I'm also reporting Precision@20 alongside it, since the very top of a ranked
queue (what gets looked at first) can behave differently from a wider top-50
cut — a metric measured at only one K can hide that difference.

'Good' here means beating a simple baseline: if pages were ranked randomly,
Precision@50 would just equal the overall declining rate (~0.54). Anything
meaningfully above that — and ideally above the hand-rule baseline from last
week — is what counts as a genuinely useful ranking.

In [20]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values

random_baseline = y.mean()
print("Random-ranking Precision@50 (= overall declining rate):", round(random_baseline, 3))
print("Random-ranking Precision@20 (= overall declining rate):", round(random_baseline, 3))

Random-ranking Precision@50 (= overall declining rate): 0.542
Random-ranking Precision@20 (= overall declining rate): 0.542


Unit of analysis: one row = one page.

Each row represents a single content page at one snapshot in time, described
by pre-decision SEO signals only — signals that were knowable before anyone
decided the page was "declining." No client names, URLs, or private queries
are shown, in line with the no-private-data rule.

The columns below are exactly the features available to rank on: how old the
content is, how long since it was last touched, how much visibility it's
getting, where it ranks, how well it converts clicks, and how long it is.
Together with the label, this is the full slice this lane operates on

In [21]:
cols = ["content_age_days", "days_since_last_update", "impressions_90d",
        "avg_position", "ctr", "word_count", "is_declining_label"]

print("One row = one page. Shape:", df[cols].shape)
df[cols].head()

One row = one page. Shape: (30000, 7)


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,1
1,445,25,15320,20.3,0.05,2481.0,1
2,141,20,12581,36.5,0.09,3515.0,1
3,463,22,11751,6.2,0.49,NaN,0
4,263,14,19140,44.0,0.13,2803.0,1


The hand rule from last week — stale (>=180 days since update) AND visible
(>=500 impressions) — applies the same two fixed thresholds to every single
page, regardless of content type, topic, or seasonality. That rigidity is
the actual weakness, even though last week's own held-out test showed the
hand rule matching or slightly beating a depth-2 tree at Precision@20/@50 on
this dataset.

So the honest claim isn't "the model already wins" — it's that a single
global cutoff can't be the right answer for every segment at once. A
7,000-word evergreen guide and a 300-word news blurb likely "go stale" on
very different timelines, but the fixed rule treats 180 days as equally
meaningful for both. A model can, in principle, learn different effective
thresholds for different slices of the data automatically, instead of
forcing one number to fit everything.

The evidence below shows the rule's rigidity directly: the same stale+visible
cutoff fires at very different rates depending on word count, even though
the rule never looks at word count at all — that blind spot is exactly what
a fixed if-statement can't see and a model could.

In [22]:
df["stale_visible"] = ((df["days_since_last_update"] >= 180) &
                        (df["impressions_90d"] >= 500)).astype(int)

print("Hand-rule 'stale+visible' flag rate, split by word_count quartile:")
print(df.groupby(pd.qcut(df["word_count"], 4))["stale_visible"].mean())

print("\nActual declining rate, same quartiles (for comparison):")
print(df.groupby(pd.qcut(df["word_count"], 4))["is_declining_label"].mean())

Hand-rule 'stale+visible' flag rate, split by word_count quartile:
word_count
(7.999, 2413.0]     0.000359
(2413.0, 2877.0]    0.000179
(2877.0, 3666.0]    0.000719
(3666.0, 9546.0]    0.001794
Name: stale_visible, dtype: float64

Actual declining rate, same quartiles (for comparison):
word_count
(7.999, 2413.0]     0.497131
(2413.0, 2877.0]    0.594701
(2877.0, 3666.0]    0.582824
(3666.0, 9546.0]    0.599498
Name: is_declining_label, dtype: float64


/tmp/ipykernel_1350/1126663904.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.qcut(df["word_count"], 4))["stale_visible"].mean())
/tmp/ipykernel_1350/1126663904.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.qcut(df["word_count"], 4))["is_declining_label"].mean())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.